# PI Few-Shot Prototypical Training

Train a prototypical multi-antenna SHARP encoder on the source PI domains, then compare K-shot target-domain prototype inference against the saved softmax baseline checkpoint.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DOPPLER_DIR = PROJECT_ROOT / "data" / "doppler_traces_pi"
SOFTMAX_CHECKPOINT_PATH = PROJECT_ROOT / "experiments" / "pi_classification" / "pi_all_persons_123_train_4_test_sharp_model_20260525_165437" / "model.pt"

RUN_GROUP = "few_shot_proto_evaluation"
RUN_NAME = "proto_multi_antenna_vs_softmax_baseline"


## Imports And Experiment Constants

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import clear_output, display
from torch.cuda import is_available
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from wifi_doppler.data.dataset import DopplerWindowDataset
from wifi_doppler.evaluation.fewshot import evaluate_kshot, window_true_labels_from_recordings
from wifi_doppler.experiments.artifacts import create_run_dir, plot_step_curves, save_checkpoint, save_figure, save_json
from wifi_doppler.models.base_model import MultiAntennaModel, SingleAntennaModel
from wifi_doppler.training.prototypical import load_cross_dataset_episode, load_episode, prototypical_loss, sample_cross_dataset_episode_indices, sample_episode_indices

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if is_available() else "cpu"
softmax_checkpoint = torch.load(SOFTMAX_CHECKPOINT_PATH, map_location=device, weights_only=False)
softmax_config = softmax_checkpoint["config"]

PERSONS = softmax_checkpoint["labels"]
TRAIN_SCENARIOS = softmax_config.get("train_scenarios", ["PI-1a", "PI-2a", "PI-3a"])
TARGET_SCENARIOS = softmax_config.get("test_scenarios", ["PI-4a"])
WINDOW_SIZE = softmax_config.get("window_size", 340)
WINDOW_STRIDE = softmax_config.get("window_stride", 30)
SOFTMAX_FUSION = softmax_config.get("fusion", "mean")

SOURCE_TRAIN_SPLIT = (0.0, 0.6)
TARGET_ENROLLMENT_SPLIT = (0.0, 0.6)
TARGET_QUERY_SPLIT = (0.6, 0.8)

TOTAL_PROTO_STEPS = 4500
EVAL_EVERY_STEPS = 100
PROTO_TEST_EPISODES_PER_EVAL = 20
LIVE_PLOT_EVERY_STEPS = 1
PROTO_N_WAY = len(PERSONS)
PROTO_K_SHOT = 5
PROTO_Q_QUERY = 10
PROTO_LR = 1e-4
EMBEDDING_FUSION = "mean"
METRIC = "cosine"

K_VALUES = [1, 3, 5, 10, 25, 50, 100]
N_TRIALS = 20
EVAL_BATCH_SIZE = 128

print("Softmax checkpoint:", SOFTMAX_CHECKPOINT_PATH)
print("Train scenarios:", TRAIN_SCENARIOS)
print("Target scenarios:", TARGET_SCENARIOS)
print("Labels:", PERSONS)
print("Device:", device)


## Build Source And Target Datasets

The prototypical model is trained only on the source domains. The target domain is split into enrollment and query windows for post-training K-shot evaluation.

In [ ]:
proto_train_dataset = DopplerWindowDataset(
    DOPPLER_DIR,
    scenarios=TRAIN_SCENARIOS,
    split=SOURCE_TRAIN_SPLIT,
    window_size=WINDOW_SIZE,
    window_stride=WINDOW_STRIDE,
    labels=PERSONS,
)

target_enrollment_dataset = DopplerWindowDataset(
    DOPPLER_DIR,
    scenarios=TARGET_SCENARIOS,
    split=TARGET_ENROLLMENT_SPLIT,
    window_size=WINDOW_SIZE,
    window_stride=WINDOW_STRIDE,
    labels=PERSONS,
)

target_query_dataset = DopplerWindowDataset(
    DOPPLER_DIR,
    scenarios=TARGET_SCENARIOS,
    split=TARGET_QUERY_SPLIT,
    window_size=WINDOW_SIZE,
    window_stride=WINDOW_STRIDE,
    labels=PERSONS,
)

proto_train_labels = window_true_labels_from_recordings(proto_train_dataset)
target_enrollment_labels = window_true_labels_from_recordings(target_enrollment_dataset)
target_query_labels = window_true_labels_from_recordings(target_query_dataset)

print("Source train windows:", len(proto_train_dataset))
print("Target enrollment windows:", len(target_enrollment_dataset))
print("Target query windows:", len(target_query_dataset))


## Train Prototypical Multi-Antenna Encoder

In [ ]:
def build_multi_antenna_model(num_classes: int, device: str | torch.device):
    model = MultiAntennaModel(SingleAntennaModel(num_classes=num_classes)).to(device)
    with torch.no_grad():
        dummy = target_enrollment_dataset[0][0].unsqueeze(0).to(device)
        _ = model.forward_antennas(dummy)
    return model
def evaluate_target_proto_episodes(model, rng):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    with torch.no_grad():
        for _ in range(PROTO_TEST_EPISODES_PER_EVAL):
            support_indices, query_indices = sample_cross_dataset_episode_indices(
                target_enrollment_labels,
                target_query_labels,
                n_way=PROTO_N_WAY,
                k_shot=PROTO_K_SHOT,
                q_query=PROTO_Q_QUERY,
                rng=rng,
            )
            episode = load_cross_dataset_episode(target_enrollment_dataset, support_indices, target_query_dataset, query_indices)
            loss, acc = prototypical_loss(
                model,
                episode,
                device=device,
                embedding_fusion=EMBEDDING_FUSION,
                metric=METRIC,
            )
            total_loss += loss.item()
            total_acc += acc

    return {
        "loss": total_loss / PROTO_TEST_EPISODES_PER_EVAL,
        "acc": total_acc / PROTO_TEST_EPISODES_PER_EVAL,
    }


def show_live_proto_curves(history):
    fig, _ = plot_step_curves(
        history,
        loss_keys=["proto_train_loss", "proto_test_loss"],
        acc_keys=["proto_train_acc", "proto_test_acc"],
        title="Prototypical training",
    )
    clear_output(wait=True)
    display(fig)
    plt.close(fig)


proto_model = build_multi_antenna_model(len(PERSONS), device)
optimizer = torch.optim.Adam(proto_model.parameters(), lr=PROTO_LR)
episode_rng = np.random.default_rng(SEED)
test_episode_rng = np.random.default_rng(SEED + 1)

proto_history = {
    "step": [],
    "proto_train_loss": [],
    "proto_train_acc": [],
    "proto_test_loss": [],
    "proto_test_acc": [],
}

progress = tqdm(range(1, TOTAL_PROTO_STEPS + 1), desc="proto training steps")
for step in progress:
    proto_model.train()
    support_indices, query_indices = sample_episode_indices(
        proto_train_labels,
        n_way=PROTO_N_WAY,
        k_shot=PROTO_K_SHOT,
        q_query=PROTO_Q_QUERY,
        rng=episode_rng,
    )
    episode = load_episode(proto_train_dataset, support_indices, query_indices)

    optimizer.zero_grad()
    loss, acc = prototypical_loss(
        proto_model,
        episode,
        device=device,
        embedding_fusion=EMBEDDING_FUSION,
        metric=METRIC,
    )
    loss.backward()
    optimizer.step()

    test_metrics = {"loss": np.nan, "acc": np.nan}
    should_eval = step == 1 or step % EVAL_EVERY_STEPS == 0 or step == TOTAL_PROTO_STEPS
    if should_eval:
        test_metrics = evaluate_target_proto_episodes(proto_model, test_episode_rng)

    proto_history["step"].append(step)
    proto_history["proto_train_loss"].append(loss.item())
    proto_history["proto_train_acc"].append(acc)
    proto_history["proto_test_loss"].append(test_metrics["loss"])
    proto_history["proto_test_acc"].append(test_metrics["acc"])

    progress.set_postfix(
        train_loss=f"{loss.item():.4f}",
        train_acc=f"{acc:.4f}",
        test_acc="nan" if np.isnan(test_metrics["acc"]) else f"{test_metrics['acc']:.4f}",
    )

    if step == 1 or step % LIVE_PLOT_EVERY_STEPS == 0 or step == TOTAL_PROTO_STEPS:
        show_live_proto_curves(proto_history)
        print(
            f"step {step:04d}/{TOTAL_PROTO_STEPS}: "
            f"proto_train_loss={loss.item():.4f} proto_train_acc={acc:.4f} "
            f"proto_test_loss={test_metrics['loss']:.4f} proto_test_acc={test_metrics['acc']:.4f}"
        )


## Load Softmax Baseline

In [ ]:
softmax_model = build_multi_antenna_model(len(PERSONS), device)
softmax_model.load_state_dict(softmax_checkpoint["model_state_dict"])
softmax_model.eval()

print("Loaded softmax baseline.")


## Zero-Shot Softmax Accuracy

In [ ]:
def evaluate_softmax_accuracy(model, dataset, device, fusion, batch_size=128):
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    correct = 0
    total = 0
    model.eval()
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x, fusion=fusion)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += y.size(0)
    return correct / total


zero_shot_softmax_acc = evaluate_softmax_accuracy(
    softmax_model,
    target_query_dataset,
    device,
    SOFTMAX_FUSION,
    batch_size=EVAL_BATCH_SIZE,
)
print(f"Zero-shot softmax accuracy on target query split: {zero_shot_softmax_acc:.4f}")


## K-Shot Prototype Inference Comparison

Both encoders use the same target-domain enrollment/query splits and the same prototype evaluator. This isolates the effect of the training objective: softmax classification vs. episodic prototypical training.

In [ ]:
softmax_fewshot_results = evaluate_kshot(
    softmax_model,
    target_enrollment_dataset,
    target_query_dataset,
    K_VALUES,
    device=device,
    n_trials=N_TRIALS,
    seed=SEED,
    batch_size=EVAL_BATCH_SIZE,
    embedding_fusion=EMBEDDING_FUSION,
    metric=METRIC,
)

proto_fewshot_results = evaluate_kshot(
    proto_model,
    target_enrollment_dataset,
    target_query_dataset,
    K_VALUES,
    device=device,
    n_trials=N_TRIALS,
    seed=SEED,
    batch_size=EVAL_BATCH_SIZE,
    embedding_fusion=EMBEDDING_FUSION,
    metric=METRIC,
)

print("K-shot target-domain query accuracy")
for k in K_VALUES:
    softmax_result = softmax_fewshot_results[k]
    proto_result = proto_fewshot_results[k]
    print(
        f"K={k:3d} | "
        f"softmax={softmax_result['mean']:.4f} +/- {softmax_result['std']:.4f} | "
        f"proto={proto_result['mean']:.4f} +/- {proto_result['std']:.4f}"
    )


## Plot And Save Artifacts

In [ ]:
def result_series(results, k_values):
    means = [results[k]["mean"] for k in k_values]
    stds = [results[k]["std"] for k in k_values]
    return means, stds


softmax_means, softmax_stds = result_series(softmax_fewshot_results, K_VALUES)
proto_means, proto_stds = result_series(proto_fewshot_results, K_VALUES)

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(K_VALUES, softmax_means, yerr=softmax_stds, marker="o", capsize=4, label="softmax-trained embeddings")
ax.errorbar(K_VALUES, proto_means, yerr=proto_stds, marker="o", capsize=4, label="prototypical embeddings")
ax.axhline(zero_shot_softmax_acc, color="tab:red", linestyle="--", label="zero-shot softmax")
ax.set_xlabel("K enrollment windows per person")
ax.set_ylabel("target query accuracy")
ax.set_ylim(0, 1)
ax.set_xticks(K_VALUES)
ax.grid(True)
ax.legend()
fig.tight_layout()
plt.show()

history_fig, _ = plot_step_curves(
    proto_history,
    loss_keys=["proto_train_loss", "proto_test_loss"],
    acc_keys=["proto_train_acc", "proto_test_acc"],
    title="Prototypical training",
)
plt.show()

run_dir = create_run_dir(PROJECT_ROOT, RUN_GROUP, RUN_NAME)
comparison_plot_path = save_figure(fig, run_dir, "pi_few_shot_proto_vs_softmax_accuracy.png")
history_plot_path = save_figure(history_fig, run_dir, "proto_training_curves.png")

proto_config = {
    "baseline_softmax_checkpoint": SOFTMAX_CHECKPOINT_PATH,
    "doppler_dir": DOPPLER_DIR,
    "train_scenarios": TRAIN_SCENARIOS,
    "target_scenarios": TARGET_SCENARIOS,
    "source_train_split": SOURCE_TRAIN_SPLIT,
    "target_enrollment_split": TARGET_ENROLLMENT_SPLIT,
    "target_query_split": TARGET_QUERY_SPLIT,
    "window_size": WINDOW_SIZE,
    "window_stride": WINDOW_STRIDE,
    "total_proto_steps": TOTAL_PROTO_STEPS,
    "eval_every_steps": EVAL_EVERY_STEPS,
    "proto_test_episodes_per_eval": PROTO_TEST_EPISODES_PER_EVAL,
    "live_plot_every_steps": LIVE_PLOT_EVERY_STEPS,
    "proto_n_way": PROTO_N_WAY,
    "proto_k_shot": PROTO_K_SHOT,
    "proto_q_query": PROTO_Q_QUERY,
    "proto_lr": PROTO_LR,
    "embedding_fusion": EMBEDDING_FUSION,
    "metric": METRIC,
    "k_values": K_VALUES,
    "n_trials": N_TRIALS,
    "seed": SEED,
}

results = {
    "zero_shot_softmax_acc": zero_shot_softmax_acc,
    "softmax_fewshot_results": softmax_fewshot_results,
    "proto_fewshot_results": proto_fewshot_results,
    "history": proto_history,
    "config": proto_config,
    "artifacts": {
        "comparison_plot": comparison_plot_path,
        "history_plot": history_plot_path,
    },
}

results_path = save_json(run_dir / "pi_few_shot_proto_vs_softmax_results.json", results)
model_path = save_checkpoint(
    proto_model,
    run_dir,
    labels=PERSONS,
    config=proto_config,
    metrics={
        "final_proto_train_loss": proto_history["proto_train_loss"][-1],
        "final_proto_train_acc": proto_history["proto_train_acc"][-1],
        "final_proto_test_loss": proto_history["proto_test_loss"][-1],
        "final_proto_test_acc": proto_history["proto_test_acc"][-1],
        "best_proto_kshot_acc": max(result["mean"] for result in proto_fewshot_results.values()),
        "best_softmax_kshot_acc": max(result["mean"] for result in softmax_fewshot_results.values()),
        "zero_shot_softmax_acc": zero_shot_softmax_acc,
    },
    history=proto_history,
    name="proto_model.pt",
)

print("Run directory:", run_dir)
print("Results:", results_path)
print("Proto model:", model_path)
print("Comparison plot:", comparison_plot_path)
print("Training curves:", history_plot_path)
